# Code GRPO HumanEval - 50-Step Controlled Experiment

Select **Runtime > Change runtime type > T4 GPU**, then run the single cell below.

The notebook evaluates the frozen model, trains a GRPO/QLoRA adapter for 50 steps, evaluates the adapter on the same held-out tasks, creates a comparison report, and downloads the complete artifact bundle automatically.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import files as colab_files

REPOSITORY = 'https://github.com/Hamza-Nadif/code-grpo-humaneval.git'
WORKDIR = Path('/content/code-grpo-humaneval-experiment')
MODEL = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
ADAPTER_DIR = Path('outputs/qwen-code-grpo-50-steps')
BASELINE_DIR = Path('results/baseline-heldout-50-step-experiment')
TRAINED_DIR = Path('results/grpo-heldout-50-step-experiment')
BUNDLE_DIR = Path('artifacts/fifty-step-experiment')

def run(command):
    print('\n$', ' '.join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], check=True)

print('STEP 1/8 - Checking GPU and storage', flush=True)
run(['nvidia-smi'])
free_gb = shutil.disk_usage('/content').free / 2**30
print(f'Free Colab storage: {free_gb:.1f} GB', flush=True)
if free_gb < 10:
    raise RuntimeError('At least 10 GB of free Colab storage is required.')

print('STEP 2/8 - Loading the latest project version', flush=True)
if WORKDIR.exists():
    run(['git', '-C', WORKDIR, 'pull', '--ff-only', 'origin', 'main'])
else:
    run(['git', 'clone', '--branch', 'main', REPOSITORY, WORKDIR])
os.chdir(WORKDIR)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'Using Git commit: {commit}', flush=True)

print('STEP 3/8 - Installing dependencies and preparing data', flush=True)
run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', 'requirements.txt', '-r', 'requirements-dev.txt',
])
run([sys.executable, 'build_training_data.py', '--output-dir', 'data'])
run([sys.executable, '-m', 'pytest', '-q'])

print('STEP 4/8 - Evaluating the frozen baseline on 22 held-out tasks', flush=True)
run([
    sys.executable, 'evaluate_baseline.py',
    '--data', 'data/humaneval_test.jsonl',
    '--backend', 'transformers',
    '--model', MODEL,
    '--quantization', '4bit',
    '--samples-per-task', '1',
    '--max-new-tokens', '128',
    '--temperature', '0',
    '--executor', 'local',
    '--allow-local-code-execution',
    '--output-dir', BASELINE_DIR,
])

print('STEP 5/8 - Training GRPO/QLoRA for 50 optimization steps', flush=True)
run([
    sys.executable, 'train_grpo.py',
    '--model', MODEL,
    '--train-data', 'data/humaneval_train.jsonl',
    '--eval-data', 'data/humaneval_validation.jsonl',
    '--quantization', '4bit',
    '--precision', 'fp16',
    '--num-generations', '2',
    '--gradient-accumulation-steps', '2',
    '--max-completion-length', '128',
    '--max-steps', '50',
    '--executor', 'local',
    '--allow-local-code-execution',
    '--output-dir', ADAPTER_DIR,
])

print('STEP 6/8 - Evaluating the trained adapter on the same tasks', flush=True)
run([
    sys.executable, 'evaluate_baseline.py',
    '--data', 'data/humaneval_test.jsonl',
    '--backend', 'transformers',
    '--model', MODEL,
    '--adapter', ADAPTER_DIR,
    '--quantization', '4bit',
    '--samples-per-task', '1',
    '--max-new-tokens', '128',
    '--temperature', '0',
    '--executor', 'local',
    '--allow-local-code-execution',
    '--output-dir', TRAINED_DIR,
])

print('STEP 7/8 - Building the before/after comparison', flush=True)
baseline = json.loads((BASELINE_DIR / 'summary.json').read_text())
trained = json.loads((TRAINED_DIR / 'summary.json').read_text())
before = baseline['metrics'].get('pass@1', 0.0)
after = trained['metrics'].get('pass@1', 0.0)
comparison = {
    'git_commit': commit,
    'model': MODEL,
    'training_steps': 50,
    'held_out_tasks': baseline['tasks'],
    'baseline_pass_at_1': before,
    'trained_pass_at_1': after,
    'absolute_difference': after - before,
    'baseline_execution_statuses': baseline['execution_statuses'],
    'trained_execution_statuses': trained['execution_statuses'],
    'scope_note': (
        'Results use the repository custom 22-task held-out split and are not an official '
        'HumanEval leaderboard score.'
    ),
}
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
(BUNDLE_DIR / 'comparison.json').write_text(json.dumps(comparison, indent=2) + '\n')
report = (
    '# GRPO 50-Step Comparison\n\n'
    f'- Model: `{MODEL}`\n'
    f'- Git commit: `{commit}`\n'
    f'- Held-out tasks: {baseline["tasks"]}\n'
    f'- Baseline pass@1: {before:.4f}\n'
    f'- Trained pass@1: {after:.4f}\n'
    f'- Difference: {after - before:+.4f}\n\n'
    'These are custom held-out experiment results, not an official leaderboard score.\n'
)
(BUNDLE_DIR / 'REPORT.md').write_text(report)
shutil.copytree(ADAPTER_DIR, BUNDLE_DIR / 'adapter', dirs_exist_ok=True)
shutil.copytree(BASELINE_DIR, BUNDLE_DIR / 'baseline', dirs_exist_ok=True)
shutil.copytree(TRAINED_DIR, BUNDLE_DIR / 'trained', dirs_exist_ok=True)

print('STEP 8/8 - Packaging and downloading all artifacts', flush=True)
archive = shutil.make_archive(
    '/content/code-grpo-humaneval-50-step-results', 'zip', BUNDLE_DIR
)
print('\nEXPERIMENT SUCCESSFUL')
print(f'Baseline pass@1: {before:.4f}')
print(f'GRPO pass@1:     {after:.4f}')
print(f'Difference:      {after - before:+.4f}')
print(f'Downloadable archive: {archive}')
print('Starting the browser download now...', flush=True)
colab_files.download(archive)
